# 개별종목 조합D — XGBoost

## 실험 목적

KOSPI200 방향이 상승·보합·하락 중 어디인지 정해졌을 때, 같은 방향일 확률이 높은
개별종목을 찾기 위한 3분류 모델입니다.

후보는 매 거래일 **KOSPI 세부 업종지수 시가총액 상위 10개 × 업종별 KOSPI 보통주
시가총액 상위 5개**로 먼저 고정합니다. 업종의 미래 방향을 따로 예측하는 구조는 아닙니다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 원천 | HF `full/daily_price_dev.parquet`, `full/index_price_dev.parquet` |
| 홀드아웃 | `20240901` 이후 접근 금지 |
| 라벨 | T일 판단 → T+1 `adj_open` 진입 → T+6 `adj_open` 평가, 종목 ±2% |
| 외부 검증 | 날짜 그룹 expanding 12폴드 |
| 최초 학습 | 750거래일 |
| 검증·gap | 폴드당 60거래일 · 직전 5거래일 제거 |
| class weight | 각 외부 폴드 내부에서 `None`과 `balanced` 재비교 |
| 선정 지표 | Accuracy·Macro F1·하락 Recall 조화평균 |

## OOS 결과

| Accuracy | Macro F1 | 하락 Recall | 핵심지표 조화평균 |
|---:|---:|---:|---:|
| 0.3921 | 0.3419 | 0.1587 | **0.2394** |

아래 셀은 저장된 실측 리포트에서 이 모델의 폴드 결과와 class weight 선택 횟수를 다시
읽습니다. 학습 구현은 `models/stock_experiment.py`, 피처·라벨은
`features/stock_model_dataset.py`가 정본입니다.


In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "reports" / "stock_feature_combinations.json").exists():
    ROOT = ROOT.parent
report_path = ROOT / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination = 'D'
model_name = 'XGBoost'

combination_report = report["combinations"][combination]
folds = pd.DataFrame(combination_report["outer_fold_results"])
display(folds.loc[folds["model"].eq(model_name)].reset_index(drop=True))

weights = pd.DataFrame(combination_report["selected_class_weight_counts"])
display(weights.loc[weights["model"].eq(model_name)].reset_index(drop=True))


,model,fold,selected_class_weight,train_dates,valid_dates,train_rows,valid_rows,train_end,valid_start,valid_end,...,down_recall,core_harmonic_mean,training_majority_class,training_majority_baseline_accuracy,validation_majority_class,validation_majority_oracle_accuracy,validation_down_rate,validation_neutral_rate,validation_up_rate,accuracy_minus_training_majority_baseline
0,XGBoost,1,balanced,750,60,35965,2880,20130402,20130410,20130705,...,0.073639,0.155467,0,0.370139,0,0.370139,0.325347,0.370139,0.304514,0.014931
1,XGBoost,2,balanced,999,60,47805,2877,20140404,20140414,20140711,...,0.056376,0.129453,0,0.474105,0,0.474105,0.258950,0.474105,0.266945,-0.003823
2,XGBoost,3,balanced,1248,60,59664,2841,20150413,20150421,20150716,...,0.147448,0.239424,0,0.332981,-1,0.372404,0.372404,0.332981,0.294615,0.022527
3,XGBoost,4,balanced,1496,60,71559,2953,20160414,20160422,20160719,...,0.067416,0.148260,0,0.412801,0,0.412801,0.331527,0.412801,0.255672,-0.004064
4,XGBoost,5,balanced,1745,60,83580,2853,20170414,20170424,20170721,...,0.140030,0.240008,0,0.418156,0,0.418156,0.230284,0.418156,0.351560,-0.001753
5,XGBoost,6,balanced,1994,60,95124,2914,20180424,20180503,20180731,...,0.130081,0.224816,0,0.391215,0,0.391215,0.337680,0.391215,0.271105,-0.025395
6,XGBoost,7,balanced,2243,60,107219,2938,20190503,20190514,20190806,...,0.120470,0.222373,0,0.461538,0,0.461538,0.347515,0.461538,0.190946,-0.038462
7,XGBoost,8,NaN,2492,60,119352,2923,20200508,20200518,20200807,...,0.268966,0.310421,0,0.314403,1,0.387958,0.297639,0.314403,0.387958,0.023606
8,XGBoost,9,balanced,2741,60,131484,2953,20210510,20210518,20210810,...,0.268805,0.335858,0,0.442262,0,0.442262,0.306129,0.442262,0.251609,-0.051812
9,XGBoost,10,NaN,2989,60,143610,3000,20220511,20220519,20220812,...,0.133333,0.228836,0,0.334333,-1,0.340000,0.340000,0.334333,0.325667,0.037333


,model,selected_class_weight,folds
0,XGBoost,balanced,8
1,XGBoost,None,4
